# Data Preprocessing Notebook

**Purpose of the File:** This notebook demonstrates the preprocessing workflow used for this project by reading NYC TLC Yellow Taxi Parquet files, inspecting schema and datatypes, checking missing values, converting the data into CSV format, and preparing a Cassandra-friendly load file.

**Repository Note:** The original Parquet files and the full generated CSV outputs are intentionally not included in this public repository because of file size considerations. To rerun this notebook, download the required TLC source files locally and update the placeholder input and output paths shown below.

### Repository Usage Notes

- Download the NYC TLC Yellow Taxi Trip Record files from: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page
- Place the downloaded files in a local folder of your choice.
- Replace the placeholder file paths in the configuration cell below before running the notebook.
- Recommended Python packages: `pandas`, `pyarrow` (or another Parquet engine), and standard library modules such as `os` and `uuid`.
- The notebook is organized into two workflows: monthly Parquet-to-CSV conversion, and January 2026 CSV-to-Cassandra-ready transformation.

### 1. Load and Inspect Source Parquet Data

This section reads the NYC TLC Yellow Taxi source data in Parquet format and reviews the initial structure, dimensions, and source datatypes before any conversion is performed.

In [ ]:
import os
import uuid

import pandas as pd

#### Configuration

Update the placeholder paths below with the local locations of the downloaded TLC files and the locations where you want the generated CSV files to be saved.

In [ ]:
# Replace the placeholder strings below with your local file paths before running the notebook.
RAW_PARQUET_FILE_JAN_2026 = "ADD_DOWNLOADED_FILE_PATH_HERE/yellow_tripdata_2026-01.parquet"
RAW_PARQUET_FILE_FEB_2026 = "ADD_DOWNLOADED_FILE_PATH_HERE/yellow_tripdata_2026-02.parquet"
MONTHLY_CSV_OUTPUT_JAN_2026 = "ADD_OUTPUT_PATH_HERE/yellow_tripdata_2026-01.csv"
MONTHLY_CSV_OUTPUT_FEB_2026 = "ADD_OUTPUT_PATH_HERE/yellow_tripdata_2026-02.csv"
MONTHLY_CSV_INPUT_JAN_2026 = "ADD_LOCAL_INPUT_CSV_PATH_HERE/yellow_tripdata_2026-01.csv"
CASSANDRA_READY_OUTPUT_JAN_2026 = "ADD_OUTPUT_PATH_HERE/yellow_tripdata_by_pickup_2026-01.csv"

In [ ]:
# -- 1. Read Parquet ---------------------------------------------------------
df = pd.read_parquet(RAW_PARQUET_FILE_FEB_2026)

In [21]:
display(df.head(10))

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,7,2026-02-01 00:05:57,2026-02-01 00:05:57,1.0,0.94,1.0,N,107,170,1,7.2,0.00,0.5,0.00,0.0,1.0,12.95,2.5,0.00,0.75
1,7,2026-02-01 00:35:58,2026-02-01 00:35:58,1.0,1.93,1.0,N,234,141,1,11.4,0.00,0.5,3.43,0.0,1.0,20.58,2.5,0.00,0.75
2,2,2026-02-01 00:08:41,2026-02-01 00:39:32,1.0,9.99,1.0,N,138,68,1,44.3,6.00,0.5,11.01,0.0,1.0,67.81,2.5,1.75,0.75
3,1,2026-02-01 00:29:06,2026-02-01 00:41:04,0.0,1.70,1.0,N,209,13,1,12.8,4.25,0.5,3.70,0.0,1.0,22.25,2.5,0.00,0.75
4,1,2026-02-01 00:53:52,2026-02-01 01:11:21,0.0,3.70,1.0,N,249,229,1,19.8,4.25,0.5,6.35,0.0,1.0,31.90,2.5,0.00,0.75
5,2,2026-02-01 00:24:29,2026-02-01 00:36:01,1.0,1.60,1.0,N,113,90,1,12.1,1.00,0.5,0.09,0.0,1.0,17.94,2.5,0.00,0.75
6,2,2026-02-01 00:40:20,2026-02-01 00:54:57,2.0,1.73,1.0,N,234,144,1,14.2,1.00,0.5,1.00,0.0,1.0,20.95,2.5,0.00,0.75
7,2,2026-02-01 00:11:48,2026-02-01 00:22:41,1.0,2.27,1.0,N,237,151,1,12.8,1.00,0.5,2.00,0.0,1.0,19.80,2.5,0.00,0.00
8,2,2026-02-01 00:02:38,2026-02-01 00:26:38,1.0,4.92,1.0,N,148,263,1,26.1,1.00,0.5,4.78,0.0,1.0,36.63,2.5,0.00,0.75
9,2,2026-02-01 00:05:56,2026-02-01 00:22:06,1.0,1.98,1.0,N,79,170,1,15.6,1.00,0.5,4.27,0.0,1.0,25.62,2.5,0.00,0.75


In [22]:
print("=" * 60)
print("SHAPE:", df.shape)                          # (rows, cols)
print("=" * 60)

SHAPE: (3399866, 20)


In [23]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3399866 entries, 0 to 3399865
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [ ]:
print("\n── Column Dtypes ──")
print(df.dtypes)


── Column Dtypes ──
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object


### 2. Standardize Datatypes and Export CSV

This section aligns the working dataframe with the expected schema, handles datetime conversion, checks missing values, and exports the cleaned dataset into CSV format for downstream use.

In [25]:
# Force consistent dtypes based on parquet schema

df["VendorID"] = df["VendorID"].astype("Int32")
df["PULocationID"] = df["PULocationID"].astype("Int32")
df["DOLocationID"] = df["DOLocationID"].astype("Int32")
df["payment_type"] = df["payment_type"].astype("Int64")

df["passenger_count"] = df["passenger_count"].astype("float64")
df["trip_distance"] = df["trip_distance"].astype("float64")
df["RatecodeID"] = df["RatecodeID"].astype("float64")
df["fare_amount"] = df["fare_amount"].astype("float64")
df["extra"] = df["extra"].astype("float64")
df["mta_tax"] = df["mta_tax"].astype("float64")
df["tip_amount"] = df["tip_amount"].astype("float64")
df["tolls_amount"] = df["tolls_amount"].astype("float64")
df["improvement_surcharge"] = df["improvement_surcharge"].astype("float64")
df["total_amount"] = df["total_amount"].astype("float64")
df["congestion_surcharge"] = df["congestion_surcharge"].astype("float64")
df["Airport_fee"] = df["Airport_fee"].astype("float64")
df["cbd_congestion_fee"] = df["cbd_congestion_fee"].astype("float64")

df["store_and_fwd_flag"] = df["store_and_fwd_flag"].astype("string")

df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

In [26]:
# Check for missing values
print(df.isna().sum())

VendorID                       0
tpep_pickup_datetime           0
tpep_dropoff_datetime          0
passenger_count          1023317
trip_distance                  0
RatecodeID               1023317
store_and_fwd_flag       1023317
PULocationID                   0
DOLocationID                   0
payment_type                   0
fare_amount                    0
extra                          0
mta_tax                        0
tip_amount                     0
tolls_amount                   0
improvement_surcharge          0
total_amount                   0
congestion_surcharge     1023317
Airport_fee              1023317
cbd_congestion_fee             0
dtype: int64


In [27]:
# -- Save monthly CSV outputs ------------------------------------------------
output_path = MONTHLY_CSV_OUTPUT_JAN_2026
output_path2 = MONTHLY_CSV_OUTPUT_FEB_2026

# Export CSV with explicit datetime format
df.to_csv(
    output_path2,
    index=False,
    date_format="%Y-%m-%d %H:%M:%S"
)

print(df.dtypes)
print(f"Saved to {output_path2}")

VendorID                          Int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag       string[python]
PULocationID                      Int32
DOLocationID                      Int32
payment_type                      Int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object
Saved to ~/Desktop/yellow_tripdata_2026-02.csv


### 3. Validate Converted CSV Output

This section re-reads the exported CSV file to confirm how datatypes appear after conversion, validates datetime parsing behavior, and reports the generated file sizes.

In [28]:
final_df = pd.read_csv(output_path2)

/var/folders/vq/h05pc2v136q74t5mxdwxjdl00000gn/T/ipykernel_41636/3490393771.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv(output_path2)


In [29]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3399866 entries, 0 to 3399865
Data columns (total 20 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   VendorID               int64  
 1   tpep_pickup_datetime   object 
 2   tpep_dropoff_datetime  object 
 3   passenger_count        float64
 4   trip_distance          float64
 5   RatecodeID             float64
 6   store_and_fwd_flag     object 
 7   PULocationID           int64  
 8   DOLocationID           int64  
 9   payment_type           int64  
 10  fare_amount            float64
 11  extra                  float64
 12  mta_tax                float64
 13  tip_amount             float64
 14  tolls_amount           float64
 15  improvement_surcharge  float64
 16  total_amount           float64
 17  congestion_surcharge   float64
 18  Airport_fee            float64
 19  cbd_congestion_fee     float64
dtypes: float64(13), int64(4), object(3)
memory usage: 518.8+ MB


In [30]:
final_df = pd.read_csv(
    output_path2,
    parse_dates=["tpep_pickup_datetime", "tpep_dropoff_datetime"]
)

final_df.info()

/var/folders/vq/h05pc2v136q74t5mxdwxjdl00000gn/T/ipykernel_41636/668272657.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  final_df = pd.read_csv(


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3399866 entries, 0 to 3399865
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int64         
 1   tpep_pickup_datetime   datetime64[ns]
 2   tpep_dropoff_datetime  datetime64[ns]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int64         
 8   DOLocationID           int64         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

In [ ]:
import os

file_path = os.path.expanduser(output_path)
file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
print(f"CSV file size for January 2026: {file_size_mb:.2f} MB")

CSV file size for January 2026: 379.46 MB


In [31]:
import os

file_path = os.path.expanduser(output_path2)
file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
print(f"CSV file size for February 2026: {file_size_mb:.2f} MB")

CSV file size for February 2026: 345.96 MB


### 4. Prepare Cassandra-Friendly Load File

This section reshapes the January 2026 CSV data into a Cassandra-ready format by standardizing column names, generating a trip identifier, deriving a pickup date for partitioning support, and reordering fields to match the intended Cassandra table design.

In [ ]:
input_file = MONTHLY_CSV_INPUT_JAN_2026
output_file = CASSANDRA_READY_OUTPUT_JAN_2026

In [4]:
df = pd.read_csv(input_file)

# Standardize column names to lowercase
df.columns = [c.lower() for c in df.columns]

# Convert datetime columns
df["tpep_pickup_datetime"] = pd.to_datetime(df["tpep_pickup_datetime"], errors="coerce")
df["tpep_dropoff_datetime"] = pd.to_datetime(df["tpep_dropoff_datetime"], errors="coerce")

/var/folders/vq/h05pc2v136q74t5mxdwxjdl00000gn/T/ipykernel_67504/2353588657.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(input_file)


In [5]:
# Remove bad rows
df = df.dropna(subset=["tpep_pickup_datetime", "pulocationid"])

# Add Cassandra-friendly fields
df["pickup_date"] = df["tpep_pickup_datetime"].dt.date
df["trip_id"] = [str(uuid.uuid4()) for _ in range(len(df))]

# Convert datetime to Cassandra-friendly timestamp string
df["tpep_pickup_datetime"] = df["tpep_pickup_datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")
df["tpep_dropoff_datetime"] = df["tpep_dropoff_datetime"].dt.strftime("%Y-%m-%d %H:%M:%S")

# Reorder columns exactly like Cassandra table
cols = [
    "pulocationid", "pickup_date", "tpep_pickup_datetime", "trip_id",
    "vendorid", "tpep_dropoff_datetime", "passenger_count", "trip_distance",
    "ratecodeid", "store_and_fwd_flag", "dolocationid", "payment_type",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "total_amount", "congestion_surcharge",
    "airport_fee", "cbd_congestion_fee"
]

df = df[cols]

In [6]:
# Analyze the final dataframe
print("Final DataFrame shape:", df.shape)
print("Final DataFrame columns:", df.columns.tolist())
print("Final DataFrame dtypes:\n", df.dtypes)

Final DataFrame shape: (3724889, 22)
Final DataFrame columns: ['pulocationid', 'pickup_date', 'tpep_pickup_datetime', 'trip_id', 'vendorid', 'tpep_dropoff_datetime', 'passenger_count', 'trip_distance', 'ratecodeid', 'store_and_fwd_flag', 'dolocationid', 'payment_type', 'fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge', 'total_amount', 'congestion_surcharge', 'airport_fee', 'cbd_congestion_fee']
Final DataFrame dtypes:
 pulocationid               int64
pickup_date               object
tpep_pickup_datetime      object
trip_id                   object
vendorid                   int64
tpep_dropoff_datetime     object
passenger_count          float64
trip_distance            float64
ratecodeid               float64
store_and_fwd_flag        object
dolocationid               int64
payment_type               int64
fare_amount              float64
extra                    float64
mta_tax                  float64
tip_amount               float64
tolls_amou

In [7]:
df.head()

,pulocationid,pickup_date,tpep_pickup_datetime,trip_id,vendorid,tpep_dropoff_datetime,passenger_count,trip_distance,ratecodeid,store_and_fwd_flag,...,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,airport_fee,cbd_congestion_fee
0,239,2026-01-01,2026-01-01 00:54:04,b6ff2ff1-7ce0-4698-a8b4-9acad42bd911,2,2026-01-01 00:59:37,1.0,0.97,1.0,N,...,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,163,2026-01-01,2026-01-01 00:34:04,e867fc03-f822-47de-a449-c2b25c104cd4,1,2026-01-01 00:39:47,0.0,0.90,1.0,N,...,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,43,2026-01-01,2026-01-01 00:57:06,d6583907-e2c2-4ed0-9150-f6d04149c595,1,2026-01-01 01:05:59,0.0,1.40,1.0,N,...,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,142,2026-01-01,2026-01-01 00:15:22,2a110991-40de-464e-b70a-1207d2dc450b,2,2026-01-01 00:58:10,4.0,5.58,1.0,N,...,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,88,2026-01-01,2026-01-01 00:27:13,ecea537b-413c-419a-a962-bc4c80d66e7c,2,2026-01-01 00:40:43,0.0,2.16,1.0,N,...,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [8]:
df.to_csv(output_file, index=False)

print("Rows prepared:", len(df))
print("Saved:", output_file)

Rows prepared: 3724889
Saved: ~/Desktop/yellow_tripdata_by_pickup_2026-01.csv


In [9]:
import os

file_path = os.path.expanduser(output_file)
file_size_mb = os.path.getsize(file_path) / (1024 * 1024)
print(f"CSV file size for January 2026: {file_size_mb:.2f} MB")

CSV file size for January 2026: 549.97 MB
